#### Destilación Cuantizada (Quantized Distillation)

In [1]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import gc

FEATURES_SELECCIONADAS = [
    'iat', 'rst_count', 'urg_count', 'number', 'variance', 'tot_size',
    'max', 'header_length', 'flow_duration', 'weight', 'rate', 'duration',
    'protocol_type', 'syn_flag_number', 'fin_count', 'syn_count',
    'rst_flag_number', 'ack_count'
]
NOMBRE_CLASE_BENIGNA = 'BenignTraffic'

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


TF version: 2.10.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
df = pd.read_feather('df_binary_balanced.feather')
df['label_binario'] = (df['label'] != NOMBRE_CLASE_BENIGNA).astype(int)
print('Shape:', df.shape)
print(df['label_binario'].value_counts())


Shape: (1976752, 48)
label_binario
0    988376
1    988376
Name: count, dtype: int64


In [3]:
X = df[FEATURES_SELECCIONADAS].values
y = df['label_binario'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=2/9, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

del df; gc.collect()

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')
print(f'Features: {X_train.shape[1]}')


Train: 1,383,725  Val: 395,351  Test: 197,676
Features: 18


In [4]:
def focal_loss(gamma=2.0, alpha=0.75):
    """
    Focal Loss binaria.
    gamma > 0 enfoca en ejemplos dificiles.
    alpha = peso clase positiva (malicioso).
    NO combinar con class_weight.
    """
    def _loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1.0 - y_true) * tf.math.log(1.0 - y_pred)
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
        return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * bce)
    return _loss

# FIX 1 — Instanciar focal_loss UNA SOLA VEZ y reutilizar en todo el notebook.
# focal_loss() es un closure que retorna la funcion interna _loss.
# Reutilizar la misma instancia evita ambiguedad al cargar el modelo y compilar.
loss_fn = focal_loss(gamma=2.0, alpha=0.75)


In [6]:
import tensorflow as tf
from qkeras import QDense, QActivation

# 1. Cargar el Profesor (Tu modelo base congelado)
teacher_model = tf.keras.models.load_model('mlp_binario_v2_exp2_focalloss.h5', custom_objects={'_loss': loss_fn})
teacher_model.trainable = False

# 2. Definir el Estudiante (Mucho más pequeño y ya cuantizado a 8-bits)
student_model = tf.keras.Sequential([
    QDense(32, input_shape=(18,), kernel_quantizer="quantized_bits(8,0,alpha=1)", name='qdense_1'),
    tf.keras.layers.BatchNormalization(),
    QActivation("quantized_relu(4,0)"),
    QDense(16, kernel_quantizer="quantized_bits(8,0,alpha=1)", name='qdense_2'),
    tf.keras.layers.BatchNormalization(),
    QActivation("quantized_relu(4,0)"),
    QDense(1, activation='sigmoid', kernel_quantizer="quantized_bits(8,0,alpha=1)", name='output')
])

# 3. Clase personalizada de Destilación
class Distiller(tf.keras.Model):
    def __init__(self, student, teacher):
        super(Distiller, self).__init__()
        self.teacher = teacher
        self.student = student

    def compile(self, optimizer, metrics, student_loss_fn, distillation_loss_fn, alpha=0.1, temperature=3):
        super(Distiller, self).compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn # Tu focal loss
        self.distillation_loss_fn = distillation_loss_fn # Usualmente Kullback-Leibler o MSE
        self.alpha = alpha
        self.temperature = temperature

    def train_step(self, data):
        x, y = data

        # Pasar los datos por el profesor (sin gradientes)
        teacher_predictions = self.teacher(x, training=False)

        with tf.GradientTape() as tape:
            # Pasar los datos por el estudiante
            student_predictions = self.student(x, training=True)

            # 1. Pérdida del estudiante contra las etiquetas reales (Focal Loss)
            student_loss = self.student_loss_fn(y, student_predictions)

            # 2. Pérdida de destilación (Estudiante intentando imitar al Profesor)
            # Como es clasificación binaria, MSE entre las salidas sigmoidales funciona muy bien.
            distillation_loss = self.distillation_loss_fn(teacher_predictions, student_predictions)

            # 3. Pérdida combinada ponderada por alpha
            loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss

        # Calcular y aplicar gradientes al ESTUDIANTE
        trainable_vars = self.student.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        # Actualizar métricas
        self.compiled_metrics.update_state(y, student_predictions)
        return {m.name: m.result() for m in self.metrics}

    def call(self, x):
        return self.student(x)

# 4. Instanciar, compilar y entrenar
distiller = Distiller(student=student_model, teacher=teacher_model)
distiller.compile(
    optimizer=tf.keras.optimizers.Adam(),
    metrics=[tf.keras.metrics.AUC()],
    student_loss_fn=loss_fn,
    distillation_loss_fn=tf.keras.losses.MeanSquaredError(),
    alpha=0.5 # Balance entre imitar al profesor y acertar las etiquetas reales
)

distiller.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

# 5. Guardar el modelo estudiante final
student_model.save('modelo_estudiante_qat.h5')

Epoch 1/10
43242/43242 [==============================] - 180s 4ms/step - auc_1: 0.9969 - val_loss: 1.1004e-04 - val_auc_1: 0.9981
Epoch 2/10
43242/43242 [==============================] - 178s 4ms/step - auc_1: 0.9975 - val_loss: 1.1004e-04 - val_auc_1: 0.9981
Epoch 3/10
43242/43242 [==============================] - 177s 4ms/step - auc_1: 0.9976 - val_loss: 1.1004e-04 - val_auc_1: 0.9983
Epoch 4/10
43242/43242 [==============================] - 174s 4ms/step - auc_1: 0.9976 - val_loss: 1.1004e-04 - val_auc_1: 0.9983
Epoch 5/10
43242/43242 [==============================] - 174s 4ms/step - auc_1: 0.9976 - val_loss: 1.1004e-04 - val_auc_1: 0.9983
Epoch 6/10
43242/43242 [==============================] - 174s 4ms/step - auc_1: 0.9976 - val_loss: 1.1004e-04 - val_auc_1: 0.9983
Epoch 7/10
43242/43242 [==============================] - 173s 4ms/step - auc_1: 0.9976 - val_loss: 1.1004e-04 - val_auc_1: 0.9983
Epoch 8/10
43242/43242 [==============================] - 166s 4ms/step - auc_1: 0.